In [ ]:
# Instala as bibliotecas necessárias direto no contêiner do Jupyter
%pip install kafka-python boto3 pillow --quiet

import json
import time
from datetime import datetime
from kafka import KafkaConsumer
import boto3

In [ ]:
# Configura o cliente do MinIO (Simulando o AWS S3)
s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='root-minio',
    aws_secret_access_key='root12345678',
    region_name='us-east-1'
)

# Garante que o bucket do datalake existe
BUCKET_NAME = "datalake"
try:
    s3_client.create_bucket(Bucket=BUCKET_NAME)
except Exception:
    pass # Se já existir, segue o jogo

# 2. Configura o Consumidor do Kafka
consumer = KafkaConsumer(
    'cdc.public.pedidos', # Tópico gerado pelo Debezium
    bootstrap_servers=['kafka:29092'],
    auto_offset_reset='earliest', # Começa do início do tópico
    enable_auto_commit=True,      # Confirma a leitura automaticamente
    group_id='grupo-python-consumer',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')) # Transforma bytes em dicionário Python
)

print("✅ Conexões estabelecidas! Pronto para consumir...")

In [ ]:
buffer_mensagens = []
ULTIMO_FLUSH = time.time()
LIMITE_MENSAGENS = 5
INTERVALO_TEMPO_SEGUNDOS = 10

print("Aguardando eventos do Postgres/Debezium (Pressione 'Stop' no Jupyter para parar)...")

try:
    # O consumer.poll busca novas mensagens sem travar o loop infinitamente
    while True:
        # Busca mensagens no Kafka (espera até 1 segundo por novos dados)
        mensagens_recebidas = consumer.poll(timeout_ms=1000)
        
        for tp, msgs in mensagens_recebidas.items():
            for msg in msgs:
                payload_completo = msg.value
                
                # O Debezium joga o dado real dentro de payload -> after
                if payload_completo and 'payload' in payload_completo and payload_completo['payload']['after']:
                    dados_pedido = payload_completo['payload']['after']
                    operacao = payload_completo['payload']['op'] # c = insert, u = update
                    
                    dados_pedido['tipo_operacao_cdc'] = operacao
                    buffer_mensagens.append(dados_pedido)
                    print(f"📥 Evento capturado! Pedido: {dados_pedido['id_pedido']} | Status: {dados_pedido['status']}")

        # Regra de negócio do Streaming: Hora de descarregar (Flush) para o Data Lake?
        tempo_decorrido = time.time() - ULTIMO_FLUSH
        
        if len(buffer_mensagens) >= LIMITE_MENSAGENS or (tempo_decorrido >= INTERVALO_TEMPO_SEGUNDOS and len(buffer_mensagens) > 0):
            
            # Cria um nome de arquivo único baseado no timestamp atual
            timestamp_str = datetime.now().strftime("%Y%m%d-%H%M%S")
            nome_arquivo = f"live/pedidos/pedidos_batch_{timestamp_str}.jsonl"
            
            # Converte a lista de dicionários em formato JSON Lines (um JSON por linha)
            conteudo_jsonl = "\n".join([json.dumps(m) for m in buffer_mensagens])
            
            # Faz o upload direto para a memória do MinIO (S3)
            s3_client.put_object(
                Bucket=BUCKET_NAME,
                Key=nome_arquivo,
                Body=conteudo_jsonl.encode('utf-8')
            )
            
            print(f"[FLUSH] {len(buffer_mensagens)} mensagens persistidas no MinIO: {nome_arquivo}")
            
            # Limpa o buffer para o próximo ciclo
            buffer_mensagens.clear()
            ULTIMO_FLUSH = time.time()

except KeyboardInterrupt:
    print("\nSinal de interrupção recebido! Iniciando fechamento seguro...")
    
    # SALVA-VIDAS: Se sobrou alguma mensagem no buffer, faz o último upload
    if len(buffer_mensagens) > 0:
        print(f"Atenção: {len(buffer_mensagens)} mensagens remanescentes encontradas no buffer.")
        print("Gravando dados órfãos no MinIO antes de fechar...")
        
        timestamp_str = datetime.now().strftime("%Y%m%d-%H%M%S")
        nome_arquivo = f"live/pedidos/pedidos_final_{timestamp_str}.jsonl"
        conteudo_jsonl = "\n".join([json.dumps(m) for m in buffer_mensagens])
        
        s3_client.put_object(
            Bucket=BUCKET_NAME,
            Key=nome_arquivo,
            Body=conteudo_jsonl.encode('utf-8')
        )
        print(f"Sucesso! Último arquivo salvo: {nome_arquivo}")
    else:
        print("Buffer estava vazio. Nenhum dado foi perdido.")

finally:
    # Agora sim, fecha a porteira do Kafka com segurança
    consumer.close()
    print("🔌 Conexão com o Kafka encerrada com sucesso. Sistema offline.")